# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

In [1]:
import polars as pl

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [3]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")
scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
8359,5431,0,"""VUSAPWT1B2""",0.5,1,1,1,null,null,"""CCDC33""","""POLR1C""",null,null,"""ENSG00000171453""",null,null,null,null,"""POLR1C"""
13662,6137,0,"""VUSAPWT1B2""",0.0,1,1,1,null,null,"""CYSRT1""","""PITX1""",null,null,"""ENSG00000069011""",null,null,null,null,"""PITX1"""
14441,52920,2687,"""VUSAPWT1B1""",0.8125,1,1,1,1.0,1.0,"""GGA3""","""LITAF""","""Pro34Thr""","""100C>A""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,2.0,"""LITAF_Pro34Thr"""
70933,71001,205674,"""VUSAPWT6B1""",0.9375,6,1,1,1.0,1.0,"""PRR4""","""UBQLN2""","""Pro497Ser""","""1489C>T""","""ENSG00000188021""","""CEGS2""","""1_Pathogenic""",null,1.0,"""UBQLN2_Pro497Ser"""
71163,52920,2686,"""VUSAPWT2B1""",0.3125,2,1,1,1.0,1.0,"""KCNIP1""","""LITAF""","""Pro39Ser""","""115C>T""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,1.0,"""LITAF_Pro39Ser"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4576,4062,201433,"""VUSAPWT2B1""",0.125,2,1,1,1.0,1.0,"""BCL2L13""","""PLP1""","""Leu189Pro""","""566T>C""","""ENSG00000123560""","""CEGS2""","""4_VUS""",null,1.0,"""PLP1_Leu189Pro"""
12303,3535,201190,"""VUSAPWT6B1""",0.75,6,1,1,1.0,1.0,"""GORAB""","""EMD""","""Met1Val""","""1A>G""","""ENSG00000102119""","""CEGS2""","""1_Pathogenic""",9.5000e-7,2.0,"""EMD_Met1Val"""
13286,71001,205675,"""VUSAPWT2B1""",0.6875,2,1,1,1.0,1.0,"""AGR3""","""UBQLN2""","""Pro506Thr""","""1516C>A""","""ENSG00000188021""","""CEGS2""","""1_Pathogenic""",null,1.0,"""UBQLN2_Pro506Thr"""


In [4]:
scores_pd = scores.to_pandas()
scores_pd[
    (scores_pd.ad_orf_id == '12626.0') &
    (scores_pd.db_orf_id == '9905.0') #&
    # (scores_pd.db_mut_id == 215290)
    ]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0


In [5]:
import os
import shutil

# Remove all files in subfolders edgotyping_profiles
profile_dirs = [
    # f"{outputs_dir}/edgotype_profiles/manual",
    f"{outputs_dir}/edgotype_profiles/auto_norm", 
    f"{outputs_dir}/edgotype_profiles/auto_summed"
]

for profile_dir in profile_dirs:
    if os.path.exists(profile_dir):
        # Remove all files in the directory
        for filename in os.listdir(profile_dir):
            file_path = os.path.join(profile_dir, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')
        print(f"Cleared all files from {profile_dir}")
    else:
        print(f"Directory {profile_dir} does not exist")

Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_norm
Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_summed


In [6]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

# print(scores)

In [7]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/manual/orf_id_{gene}.csv")

shape_summary = pl.DataFrame(shape_summary)

In [8]:
## Add a perturbation category for each edge to use for the N2H validation

# Extract WT scores
wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# Merge WT scores back onto the main df
scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# Compute allele/WT ratio
scores_with_wt = scores_with_wt.with_columns(
    (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
)

# Determine if the allele is perturbed - binary output
threshold = 0.5
scores_with_wt = scores_with_wt.with_columns(
    ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
)

# Build perturbation_classes expression
bins = [
    (1.0, float("inf"), "increased edge"),
    (0.8, 1.0, "similar"),
    (0.6, 0.8, "slightly perturbed"),
    (0.4, 0.6, "perturbed"),
    (0.2, 0.4, "highly perturbed"),
    (0.0, 0.2, "very highly perturbed"),
]

perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

for lower, upper, label in bins:
    perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




scores_with_wt = scores_with_wt.with_columns(
    perturbation_expr.alias("perturbation_classes")
)

scores_with_perturbed = scores_with_wt.select("*")


pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

bins_2 = [
    (float("inf"), 1.0, "increased edge"),
    (1, 0.7, 'slightly perturbed'),
    (0.7, 0.3, 'perturbed'),
    (0.3, 0, 'highly perturbed')
]

for upper, lower, label in bins_2:
    pert2 = pert2.when(
        (pl.col("ratio") <= upper) &
        (pl.col("ratio") >  lower)
    ).then(pl.lit(label))

pert2 = pert2.otherwise(pl.lit("unclassified"))

scores_with_perturbed = scores_with_wt.with_columns(
    perturbation_classes_2 = pert2.alias("perturbation_classes_2")
)



cols_to_keep = ['ad_orf_id',
 'db_orf_id',
 'db_mut_id',
 'normalized_score',
 'ad_symbol',
 'symbol',
 'clinvar_clnsig_clean',
 'normalized_score_wt',
 'ratio',
 'perturbed',
 'perturbation_classes',
 "perturbation_classes_2"]

scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")

## Examine profile characteristics

In [9]:
# Number of profiled alleles with more than 5 interactors
hq_orfs = shape_summary.filter(pl.col("n_interactors") >= 5)
num_orfs = hq_orfs.shape[0]
print(num_orfs)

print(hq_orfs.select(pl.col("n_alleles")).sum() - num_orfs)

25
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 255       │
└───────────┘


In [10]:
# Total number of profiled alleles
prof_genes = shape_summary.shape[0]
print(prof_genes)

print(shape_summary.select(pl.col("n_alleles")).sum() - prof_genes)

84
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 659       │
└───────────┘


# Automated

In [11]:
import polars as pl

In [12]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [13]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["summed_consensus_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,summed_consensus_score,wt_summed_score,assay_id,ppi_group,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,i64,str,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
6552,2713,200868,"""VUSAPWT1B1""",2345.925595,5631.520448,1,"""6552_2713_200868_VUSAPWT1B1""",1,1,1.0,1.0,"""MAGEA8""","""MLH1""","""Asp485Glu""","""1455T>A""","""ENSG00000076242""","""CEGS2""",null,null,null,"""MLH1_Asp485Glu"""
54784,3689,214825,"""VUSAPWT6B1""",7522.243843,6082.385853,6,"""54784_3689_214825_VUSAPWT6B1""",1,1,1.0,1.0,"""GCKR""","""GCK""","""Gly175Arg""","""523G>C""","""ENSG00000106633""","""CEGS2""","""1_Pathogenic""",null,3.0,"""GCK_Gly175Arg"""
12260,4062,0,"""VUSAPWT1B2""",1313.251003,1313.251003,1,"""12260_4062_0_VUSAPWT1B2""",1,1,null,null,"""STX1A""","""PLP1""",null,null,"""ENSG00000123560""",null,null,null,null,"""PLP1"""
11132,52920,2678,"""VUSAPWT1B1""",1550.150348,5141.736064,1,"""11132_52920_2678_VUSAPWT1B1""",1,1,1.0,1.0,"""EPN1""","""LITAF""","""Ala111Gly""","""332C>G""","""ENSG00000189067""","""RC4""","""1_Pathogenic""",null,2.0,"""LITAF_Ala111Gly"""
100011278,55855,0,"""VUSAPWT1B2""",6935.240693,6935.240693,1,"""100011278_55855_0_VUSAPWT1B2""",1,1,null,null,"""KRT16""","""KRT2""",null,null,"""ENSG00000172867""",null,null,null,null,"""KRT2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3598,5066,201831,"""VUSAPWT1B2""",7313.172851,4738.789668,1,"""3598_5066_201831_VUSAPWT1B2""",1,1,1.0,2.0,"""DAZAP2""","""BAG3""","""Leu462Pro""","""1385T>C""","""ENSG00000151929""","""CEGS2""","""1_Pathogenic""",null,1.0,"""BAG3_Leu462Pro"""
1969,513,0,"""VUSAPWT6B1""",3411.198785,3411.198785,6,"""1969_513_0_VUSAPWT6B1""",1,1,null,null,"""ARMC12""","""FA2H""",null,null,"""ENSG00000103089""",null,null,null,null,"""FA2H"""
11567,52920,218637,"""VUSAPWT1B1""",5575.379261,8494.115307,1,"""11567_52920_218637_VUSAPWT1B1""",1,1,1.0,1.0,"""TAX1BP1""","""LITAF""","""Leu122Val""","""364C>G""","""ENSG00000189067""","""Edgotyping3""","""1_Pathogenic""",null,null,"""LITAF_Leu122Val"""


In [14]:
# # Create profiles for each profiled gene

# profiles = {}
# shape_summary = []
# for gene in genes:
#     gene_scores = scores.filter(pl.col("db_orf_id") == gene)
#     profile = gene_scores.pivot(
#                 values="normalized_score",
#                 index="ad_orf_id",
#                 on="db_mut_id"
#             )
#     profiles[gene] = profile

#     shape_summary.append({
#         "gene": gene,
#         "n_interactors": profile.height,
#         "n_alleles": profile.width
#     })

#     profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_norm/orf_id_{gene}_auto_normalized.csv")

# shape_summary = pl.DataFrame(shape_summary)

In [15]:
# # Create profiles for each profiled gene

# profiles = {}
# shape_summary = []
# for gene in genes:
#     gene_scores = scores.filter(pl.col("db_orf_id") == gene)
#     profile = gene_scores.pivot(
#                 values="adjusted_score",
#                 index="ad_orf_id",
#                 on="db_mut_id"
#             )
#     profiles[gene] = profile

#     shape_summary.append({
#         "gene": gene,
#         "n_interactors": profile.height,
#         "n_alleles": profile.width
#     })

#     profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

# shape_summary = pl.DataFrame(shape_summary)

In [16]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="summed_consensus_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)